Experimento 3: ResNet-50 com Tratamento de Desbalanceamento

Objetivo: Implementar a arquitetura ResNet-50 utilizando Transfer Learning, conforme sugerido pela orientação. Este experimento foca em superar as limitações encontradas no modelo baseline e na VGG16, atacando o problema de desbalanceamento de classes através de duas técnicas:

Data Augmentation: Para aumentar a diversidade de amostras durante o treino.
2. Class Weights: Para penalizar de forma mais rigorosa o erro nas classes minoritárias (como 'Disgust'), que representam menos de 2% do dataset FER2013.


Referência Técnica: ResNet-50 utiliza conexões residuais ("skip connections") que ajudam a mitigar o problema do gradiente vanishing em redes profundas

In [1]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import ResNet50
import numpy as np
import os

# 1. Configuração de Dados (RGB para ResNet)
tamanho_imagem = (48, 48)
tamanho_lote = 32

dados_treino = tf.keras.utils.image_dataset_from_directory(
    './fer2013/train',
    color_mode="rgb",
    image_size=tamanho_imagem,
    batch_size=tamanho_lote,
    label_mode='int'
)

dados_teste = tf.keras.utils.image_dataset_from_directory(
    './fer2013/test',
    color_mode="rgb",
    image_size=tamanho_imagem,
    batch_size=tamanho_lote,
    label_mode='int'
)

# 2. Cálculo de Class Weights (Resolvendo o desbalanceamento)
# Contamos os arquivos em cada pasta para calcular o peso matemático
classes = sorted(os.listdir('./fer2013/train'))
contagem = [len(os.listdir(f'./fer2013/train/{c}')) for c in classes]
total = sum(contagem)
n_classes = len(classes)

# Fórmula: peso = total / (n_classes * contagem_da_classe)
pesos_ajustados = {i: total / (n_classes * count) for i, count in enumerate(contagem)}
print(f"Pesos calculados para equilibrar as classes: {pesos_ajustados}")

# 3. Arquitetura com Data Augmentation e ResNet-50
data_augmentation = models.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
])

base_model = ResNet50(weights='imagenet', include_top=False, input_shape=(48, 48, 3))
base_model.trainable = False # Congelamos a base para Transfer Learning

modelo = models.Sequential([
    data_augmentation,
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.5),
    layers.Dense(len(classes), activation='softmax')
])

modelo.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# 4. Treinamento
historico = modelo.fit(
    dados_treino,
    validation_data=dados_teste,
    epochs=10, # Aumentamos um pouco para a ResNet estabilizar
    class_weight=pesos_ajustados
)

Found 28709 files belonging to 7 classes.
Found 7178 files belonging to 7 classes.
Pesos calculados para equilibrar as classes: {0: 1.0266046844269623, 1: 9.406618610747051, 2: 1.0010460615781582, 3: 0.5684387684387684, 4: 0.8260394187886635, 5: 0.8491274770777877, 6: 1.293372978330405}
Epoch 1/10
898/898 [==============================] - 105s 113ms/step - loss: 2.7365 - accuracy: 0.2375 - val_loss: 1.8396 - val_accuracy: 0.3201
Epoch 2/10
898/898 [==============================] - 106s 118ms/step - loss: 2.2257 - accuracy: 0.2695 - val_loss: 1.9672 - val_accuracy: 0.3242
Epoch 3/10
898/898 [==============================] - 108s 120ms/step - loss: 2.2259 - accuracy: 0.2662 - val_loss: 1.8043 - val_accuracy: 0.3278
Epoch 4/10
898/898 [==============================] - 105s 116ms/step - loss: 2.2302 - accuracy: 0.2710 - val_loss: 1.9779 - val_accuracy: 0.2984
Epoch 5/10
898/898 [==============================] - 114s 127ms/step - loss: 2.2163 - accuracy: 0.2699 - val_loss: 1.8515 - val